# 0. Setup & Environment Verification

This notebook configures the environment for training on local GPU or CPU.

**Features:**
- TensorFlow Installation & GPU Setup
- GPU Memory Management (mit Fallback auf CPU)
- Dependencies Verification
- Path Configuration
- System Information

## 0.1 Import Dependencies

In [1]:
import sys
import os
from pathlib import Path

# TensorFlow
import tensorflow as tf
import numpy as np

# Project
sys.path.insert(0, str(Path.cwd().parent))
import config

print(f"Python Version: {sys.version}")
print(f"TensorFlow Version: {tf.__version__}")
print(f"NumPy Version: {np.__version__}")

Python Version: 3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)]
TensorFlow Version: 2.21.0
NumPy Version: 2.5.1


## 0.2 GPU/CPU Configuration

Automatic GPU detection with fallback to CPU

In [2]:
# ============================================================================
# GPU SETUP with CPU Fallback
# ============================================================================

def setup_gpu_memory():
    """Configure GPU memory with a fallback to CPU"""

    if not config.GPUConfig.ENABLE_GPU:
        print("\n  GPU is disabled (config.GPUConfig.ENABLE_GPU = False)")
        print("    Using CPU for training")
        os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
        return False

    # Try to find GPUs
    gpus = tf.config.experimental.list_physical_devices('GPU')

    if not gpus:
        print("\n  No GPU found!")
        print("    Fallback to CPU")
        return False

    print(f"\n✓ {len(gpus)} GPU(s) found:")
    for i, gpu in enumerate(gpus):
        print(f"  GPU {i}: {gpu}")

    # Set Memory Growth to prevent OOM errors
    try:
        if config.GPUConfig.GPU_MEMORY_GROWTH:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            print("  ✓ GPU Memory Growth enabled (dynamic memory allocation)")

        # Optional: Limit GPU memory
        if config.GPUConfig.GPU_MEMORY_LIMIT_MB:
            gpu_limit = config.GPUConfig.GPU_MEMORY_LIMIT_MB
            for gpu in gpus:
                tf.config.experimental.set_virtual_device_configuration(
                    gpu,
                    [tf.config.experimental.VirtualDeviceConfiguration(
                        memory_limit=gpu_limit
                    )]
                )
            print(f"  ✓ GPU Memory limited to {gpu_limit} MB")

        print("\n✓ GPU configured and ready for use")
        return True

    except Exception as e:
        print(f"\n  Error during GPU configuration: {e}")
        print("    Fallback to CPU")
        return False

# Execute GPU setup
gpu_available = setup_gpu_memory()


  No GPU found!
    Fallback to CPU


## 0.3 System Information

In [3]:
print("\n" + "="*80)
print("SYSTEM INFORMATION")
print("="*80)

print(f"\nTensorFlow Build Infos:")
print(f"  CUDA available: {tf.test.is_built_with_cuda()}")
print(f"  cuDNN available: {tf.test.is_built_with_gpu_support()}")

print(f"\nLogical & Physical Devices:")
logical_gpus = tf.config.list_logical_devices('GPU')
logical_cpus = tf.config.list_logical_devices('CPU')
print(f"  CPUs: {len(logical_cpus)}")
print(f"  GPUs: {len(logical_gpus)}")

if logical_gpus:
    for gpu in logical_gpus:
        print(f"    - {gpu}")

print(f"\nBuild Information:")
print(f"  TensorFlow Version: {tf.__version__}")
print(f"  Python Version: {sys.version.split()[0]}")

print("\n" + "="*80)


SYSTEM INFORMATION

TensorFlow Build Infos:
  CUDA available: False
  cuDNN available: False

Logical & Physical Devices:
  CPUs: 1
  GPUs: 0

Build Information:
  TensorFlow Version: 2.21.0
  Python Version: 3.13.3



## 0.4 Path Configuration Verification

In [4]:
print("\n" + "="*80)
print("PATH CONFIGURATION")
print("="*80)

print(f"\nProject Root: {config.PROJECT_ROOT}")
print(f"  ✓ Exists: {config.PROJECT_ROOT.exists()}")

print(f"\nData Directories:")
counts = {}

for name, path in [
    ('Anchor', config.ANCHOR_PATH),
    ('Positive', config.POSITIVE_PATH),
    ('Negative', config.NEGATIVE_PATH),
]:
    exists = path.exists()
    count = len(list(path.glob('*.*'))) if exists else 0
    counts[name] = count
    status = "✓" if exists else "✗"
    print(f"  {status} {name}: {path.name} ({count} files)")

print(f"\nOutput Directories:")
for name, path in [
    ('Checkpoints', config.CHECKPOINT_DIR),
    ('Logs', config.LOG_DIR),
]:
    exists = path.exists()
    status = "✓" if exists else "○"
    print(f"  {status} {name}: {path.name}")

print("\n" + "="*80)


PATH CONFIGURATION

Project Root: C:\Users\angel\Documents\AI Master3.Semester\Doyourecognizeme
  ✓ Exists: True

Data Directories:
  ✓ Anchor: anchor (523 files)
  ✓ Positive: positive (523 files)
  ✓ Negative: negative (13233 files)

Output Directories:
  ✓ Checkpoints: checkpoints
  ✓ Logs: logs



## 0.5 Dependencies Verification

In [5]:
import importlib

def check_module(module_name):
    """Check if module is installed"""
    try:
        mod = importlib.import_module(module_name)
        ver = getattr(mod, '__version__', 'unknown')
        print(f"  ✓ {module_name}: {ver}")
        return True
    except ImportError:
        print(f"  ✗ {module_name}: NOT INSTALLED")
        return False

print("\nDependencies:")
required_modules = [
    'tensorflow',
    'cv2',
    'numpy',
    'matplotlib',
    'sklearn',
    'PIL',
]

for mod in required_modules:
    check_module(mod)

print("\n✓ Setup completed!")


Dependencies:
  ✓ tensorflow: 2.21.0
  ✓ cv2: 4.10.0
  ✓ numpy: 2.5.1
  ✓ matplotlib: 3.11.0
  ✓ sklearn: 1.9.0
  ✓ PIL: 12.3.0

✓ Setup completed!


## 0.6 Configuration Summary

In [6]:
print("\n" + "="*80)
print("TRAINING CONFIGURATION")
print("="*80)

print(f"\nHyperparameters:")
print(f"  Batch Size: {config.BATCH_SIZE}")
print(f"  Learning Rate: {config.LEARNING_RATE}")
print(f"  Epochs: {config.EPOCHS}")
print(f"  Early Stopping Patience: {config.EARLY_STOPPING_PATIENCE}")

print(f"\nAugmentation:")
print(f"  Brightness Delta: {config.AugmentationConfig.BRIGHTNESS_MAX_DELTA}")
print(f"  Contrast Range: [{config.AugmentationConfig.CONTRAST_LOWER}, {config.AugmentationConfig.CONTRAST_UPPER}]")
print(f"  Saturation Range: [{config.AugmentationConfig.SATURATION_LOWER}, {config.AugmentationConfig.SATURATION_UPPER}]")
print(f"  Flip Probability: {config.AugmentationConfig.FLIP_PROB*100:.0f}%")

print(f"\nModel:")
print(f"  Image Size: {config.IMG_SIZE}x{config.IMG_SIZE}x{config.IMG_CHANNELS}")
print(f"  Embedding Dimensions: {config.EmbeddingConfig.DENSE_UNITS}")
print(f"  Verification Threshold: {config.VERIFICATION_THRESHOLD}")

print(f"\nTraining Mode:")
print(f"  GPU Enabled: {config.GPUConfig.ENABLE_GPU}")
print(f"  GPU Available: {gpu_available}")
print(f"  Data Loading: On-The-Fly Augmentation (no disk storage)")

print("\n" + "="*80)
print("✓ Setup completed successfully!")
print("→ Next Notebook: 01_data_collection.ipynb")
print("="*80)


TRAINING CONFIGURATION

Hyperparameters:
  Batch Size: 16
  Learning Rate: 0.0001
  Epochs: 50
  Early Stopping Patience: 3

Augmentation:
  Brightness Delta: 0.3
  Contrast Range: [0.8, 1.2]
  Saturation Range: [0.8, 1.2]
  Flip Probability: 50%

Model:
  Image Size: 100x100x3
  Embedding Dimensions: 4096
  Verification Threshold: 0.35

Training Mode:
  GPU Enabled: True
  GPU Available: False
  Data Loading: On-The-Fly Augmentation (no disk storage)

✓ Setup completed successfully!
→ Next Notebook: 01_data_collection.ipynb
